In [ ]:
import pandas as pd
import numpy as np
import os
import time
from datetime import datetime

# غیرفعال کردن هشدارهای غیرضروری
import warnings
warnings.filterwarnings('ignore')

# --- تنظیمات آدرس‌ها ---
file_path = r'second_stage_inputs\G11\dsas_g11_turbine_bearings_output.xlsx'
output_filename = r'outputs\G11\dsas_g11_bearings_vibration_temp_univariate\univariate\dsas_g11_univariate_output4.xlsx'

# سنسورهای هدف
target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']

def run_daily_weekly_analysis():
    """اجرای تحلیل روزانه/هفتگی و ذخیره خروجی"""
    
    print("="*60)
    print(f"🔄 شروع تحلیل در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*60)
    
    if not os.path.exists(file_path):
        print(f"❌ خطا: فایل در مسیر زیر یافت نشد:\n{file_path}")
        return None

    try:
        # ۱. بارگذاری داده‌ها
        df = pd.read_excel(file_path, parse_dates=['date'])
        df = df.sort_values('date')
        print(f"✅ مرحله ۱: داده‌ها بارگذاری شدند. تعداد رکوردها: {len(df):,}")
        print(f"📅 بازه زمانی: {df['date'].min()} تا {df['date'].max()}")
    except Exception as e:
        print(f"❌ خطا در خواندن اکسل: {e}")
        return None

    # ۲. جداسازی بازه یک ماه اخیر برای تحلیل
    last_date = df['date'].max()
    one_month_ago = last_date - pd.Timedelta(days=30)
    df_recent = df[df['date'] >= one_month_ago].copy()
    
    print(f"📅 بازه تحلیل (یک ماه اخیر): {one_month_ago} تا {last_date}")
    print(f"📊 تعداد رکوردهای بازه تحلیل: {len(df_recent):,}")

    # ۳. محاسبات Dual EWMA (روزانه در مقابل هفتگی)
    alpha_fast = 0.22  # حساس به تغییرات ۲۴ ساعت اخیر
    alpha_slow = 0.035 # نمایانگر روند کلی ۷ روز اخیر

    results_list = []

    print(f"⏳ مرحله ۲: تحلیل روند - سریع (Alpha={alpha_fast}) و کند (Alpha={alpha_slow})")
    print("🔄 در حال پردازش سنسورها...")

    for col in target_sensors:
        if col in df_recent.columns:
            # ایجاد دیتافریم موقت
            temp_df = df_recent[['date', col]].copy()
            temp_df = temp_df.rename(columns={col: 'Raw_Value'})
            temp_df['AssetID'] = col

            # الف) میانگین متحرک سریع (بازه یک روزه)
            temp_df['Daily_EWMA_Fast'] = temp_df['Raw_Value'].ewm(alpha=alpha_fast, adjust=False).mean()

            # ب) میانگین متحرک کند (بازه هفتگی)
            temp_df['Weekly_EWMA_Slow'] = temp_df['Raw_Value'].ewm(alpha=alpha_slow, adjust=False).mean()

            # ج) محاسبه انحراف سیگنال (Gap)
            temp_df['Signal_Gap'] = temp_df['Daily_EWMA_Fast'] - temp_df['Weekly_EWMA_Slow']

            # د) محاسبه درصد انحراف نسبت به روند هفتگی
            temp_df['Deviation_Percent'] = (temp_df['Signal_Gap'] / temp_df['Weekly_EWMA_Slow']) * 100

            # انتخاب و چیدمان ستون‌ها
            cols_order = ['date', 'AssetID', 'Raw_Value', 'Daily_EWMA_Fast', 'Weekly_EWMA_Slow', 'Signal_Gap', 'Deviation_Percent']
            results_list.append(temp_df[cols_order])
            
            print(f"   ✅ {col}: {len(temp_df):,} رکورد پردازش شد")

    if not results_list:
        print("⚠️ هشدار: سنسورهای مورد نظر در فایل یافت نشدند.")
        return None

    # ۴. تجمیع نتایج
    print("🔄 مرحله ۳: تجمیع نتایج...")
    df_final_output = pd.concat(results_list, ignore_index=True)
    print(f"   ✅ تعداد کل رکوردها: {len(df_final_output):,}")

    # ۵. ذخیره در اکسل
    print("💾 مرحله ۴: ذخیره خروجی...")
    
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            df_final_output.to_excel(writer, index=False, sheet_name='Maintenance_Strategy')

        print(f"✅ فایل با موفقیت ذخیره شد: {output_filename}")
        print(f"📊 تعداد رکوردهای نهایی: {len(df_final_output):,}")
        print(f"📋 تعداد ستون‌ها: {len(df_final_output.columns)}")
        
        print("-"*50)
        print("💡 راهنمای تحلیل:")
        print("1. ستون Daily_EWMA_Fast: وضعیت لرزش/دما در حدوداً ۶ داده اخیر (امروز).")
        print("2. ستون Weekly_EWMA_Slow: روند کلی تجهیز در یک هفته اخیر.")
        print("3. اگر Deviation_Percent رشد مداوم داشته باشد، احتمالا خرابی در حال شکل‌گیری است.")
        
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل اکسل: {e}")
        return None
    
    print("="*60)
    print(f"✅ تحلیل در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} کامل شد")
    print("="*60)
    
    return df_final_output

def run_scheduler():
    """
    بررسی مداوم برای اجرا در زمان‌های مشخص (هر روز)
    """
    print("="*60)
    print("🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - تحلیل روزانه/هفتگی")
    print("="*60)
    print("⏰ زمان‌های اجرا (هر روز):")
    print("   - ساعت 10:00")
    print("   - ساعت 10:05")
    print("   - ساعت 10:10")
    print("="*60)
    print("💡 برای توقف برنامه، Ctrl+C را بزنید")
    print("="*60)
    
    last_run_time = None  # فقط برای جلوگیری از اجرای مجدد در یک زمان
    
    while True:
        try:
            now = datetime.now()
            current_time = now.strftime("%H:%M")
            
            # بررسی زمان‌های مشخص
            if current_time in ["09:09", "09:30"]:
                # فقط چک می‌کنیم که در همین زمان دوبار اجرا نشود
                if last_run_time != current_time:
                    print("\n" + "="*60)
                    print(f"⏰ زمان اجرا فرا رسید: {now.strftime('%Y-%m-%d %H:%M:%S')}")
                    print("="*60)
                    
                    # اجرای تابع اصلی
                    result = run_daily_weekly_analysis()
                    
                    if result is not None:
                        print("\n" + "="*60)
                        print("✅ اجرای زمان‌بندی شده با موفقیت کامل شد!")
                        print("="*60)
                    else:
                        print("\n" + "="*60)
                        print("❌ اجرای زمان‌بندی شده با شکست مواجه شد!")
                        print("="*60)
                    
                    # ثبت زمان اجرا
                    last_run_time = current_time
                    
                    # 10 ثانیه صبر کن تا از اجرای مجدد در همان دقیقه جلوگیری شود
                    time.sleep(10)
            
            # هر 10 ثانیه یکبار بررسی کن
            time.sleep(10)
            
        except KeyboardInterrupt:
            print("\n" + "="*60)
            print("⏹️ برنامه با دستور کاربر متوقف شد")
            print(f"⏹️ زمان توقف: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print("="*60)
            break
            
        except Exception as e:
            print(f"❌ خطا در حلقه اصلی: {e}")
            print("🔄 ادامه اجرا...")
            time.sleep(60)

# اجرای اصلی
if __name__ == "__main__":
    try:
        print("="*60)
        print("🚀 شروع برنامه تحلیل روزانه/هفتگی")
        print("="*60)
        
        # شروع زمان‌بندی
        run_scheduler()
        
    except Exception as e:
        print(f"❌ خطای غیرمنتظره: {e}")
        input("برای خروج Enter بزنید...")

🚀 شروع برنامه تحلیل روزانه/هفتگی
🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - تحلیل روزانه/هفتگی
⏰ زمان‌های اجرا (هر روز):
   - ساعت 10:00
   - ساعت 10:05
   - ساعت 10:10
💡 برای توقف برنامه، Ctrl+C را بزنید

⏰ زمان اجرا فرا رسید: 2026-07-07 09:09:02
🔄 شروع تحلیل در 2026-07-07 09:09:02
✅ مرحله ۱: داده‌ها بارگذاری شدند. تعداد رکوردها: 11,752
📅 بازه زمانی: 2021-03-16 05:33:48 تا 2026-05-04 05:16:35
📅 بازه تحلیل (یک ماه اخیر): 2026-04-04 05:16:35 تا 2026-05-04 05:16:35
📊 تعداد رکوردهای بازه تحلیل: 152
⏳ مرحله ۲: تحلیل روند - سریع (Alpha=0.22) و کند (Alpha=0.035)
🔄 در حال پردازش سنسورها...
   ✅ AssetID_9358: 152 رکورد پردازش شد
   ✅ AssetID_9359: 152 رکورد پردازش شد
   ✅ AssetID_9360: 152 رکورد پردازش شد
   ✅ AssetID_9361: 152 رکورد پردازش شد
🔄 مرحله ۳: تجمیع نتایج...
   ✅ تعداد کل رکوردها: 608
💾 مرحله ۴: ذخیره خروجی...
✅ فایل با موفقیت ذخیره شد: outputs\G11\dsas_g11_bearings_vibration_temp_univariate\univariate\dsas_g11_univariate_output4.xlsx
📊 تعداد رکوردهای نهایی: 608
📋 تعداد ستون‌ها: 7
